# Bronze to Silver - Cleaning & transforming dimension tables 
Fixing data quality issues found in the bronze layer for all dimension tables.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType, DateType, IntegerType

catalog_name = 'quickbite'

## Restaurants
**Issues to fix:**
- Spelling mistakes in `cuisine_type`
- Inconsistent city casing

In [0]:
df = spark.table(f"{catalog_name}.bronze.brz_restaurants")
df.show(5)

In [0]:
# Checking distinct cuisine types to spot anomalies
df.select("cuisine_type").distinct().orderBy("cuisine_type").show(truncate=False)

In [0]:
# Fixing spelling mistakes in cuisine_type
cuisine_fixes = {
    "Nort Indian":  "North Indian",
    "Italain":      "Italian",
    "Chinesse":     "Chinese",
    "Fast food":    "Fast Food",
    "BIRYANI":      "Biryani",
}

df_silver = df.replace(to_replace=cuisine_fixes, subset=["cuisine_type"])

# Standardising city casing
df_silver = df_silver.withColumn("city", F.initcap(F.col("city")))

# Verifying
df_silver.select("cuisine_type").distinct().orderBy("cuisine_type").show(truncate=False)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_restaurants")

print("slv_restaurants written:", spark.table(f"{catalog_name}.silver.slv_restaurants").count(), "rows")

## Customers
**Issues to fix:**
- Inconsistent `city` casing (`mumbai`, `DELHI`, `bangalore`)
- NULL values in `phone` → fill with `'Not Available'`
- Convert `signup_date` string → DateType

In [0]:
df = spark.table(f"{catalog_name}.bronze.brz_customers")
print("Total rows:", df.count())
df.show(5)

In [0]:
# Checking null counts
from pyspark.sql.functions import col, sum as _sum
df.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

In [0]:
# Fixing city casing
df_silver = df.withColumn("city", F.initcap(F.col("city")))

# Filling null phone
df_silver = df_silver.fillna("Not Available", subset=["phone"])

# Parsing signup_date
df_silver = df_silver.withColumn("signup_date", F.to_date(F.col("signup_date"), "yyyy-MM-dd"))

df_silver.show(5)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

print("slv_customers written:", spark.table(f"{catalog_name}.silver.slv_customers").count(), "rows")

## Delivery Agents
**Issues to fix:**
- `vehicle_type` has inconsistent casing (`bike`, `scooter`, `Bike`, `Scooter`)

In [0]:
df = spark.table(f"{catalog_name}.bronze.brz_delivery_agents")
df.select("vehicle_type").distinct().show()

In [0]:
# Standardising vehicle_type to Title Case
df_silver = df.withColumn("vehicle_type", F.initcap(F.col("vehicle_type")))
df_silver.select("vehicle_type").distinct().show()

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_delivery_agents")

print("slv_delivery_agents written:", spark.table(f"{catalog_name}.silver.slv_delivery_agents").count(), "rows")

## Menu Items
**Issues to fix:**
- `base_price` stored as string — cast to float

In [0]:
df = spark.table(f"{catalog_name}.bronze.brz_menu_items")
df.printSchema()
df.select("base_price").show(5)

In [0]:
df_silver = df.withColumn("base_price", F.col("base_price").cast(FloatType()))
df_silver.printSchema()

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_menu_items")

print("slv_menu_items written:", spark.table(f"{catalog_name}.silver.slv_menu_items").count(), "rows")

## Date Dimension
**Issues to fix:**
- Convert `date` string (dd-MM-yyyy) → DateType
- Remove duplicates on `date`
- `day_name` inconsistent casing → `initcap()`
- Negative `week_of_year` → `abs()`
- Enhance `quarter` and `week_of_year` labels (e.g. `Q1-2025`)

In [0]:
df = spark.table(f"{catalog_name}.bronze.brz_date")
print("Row count:", df.count())
df.show(3)

In [0]:
# Converting date string to DateType
df_silver = df.withColumn("date", F.to_date(F.col("date"), "dd-MM-yyyy"))

# Checking for duplicates
dups = df_silver.groupBy("date").count().filter(F.col("count") > 1)
print("Duplicate date rows:", dups.count())
df_silver = df_silver.dropDuplicates(["date"])
print("Rows after dedup:", df_silver.count())

In [0]:
# Standardising day_name casing
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))

# Fixing negative week_of_year
df_silver = df_silver.withColumn("week_of_year", F.abs(F.col("week_of_year")))

# Enhancing quarter label: 1 → Q1-2025
df_silver = df_silver.withColumn(
    "quarter",
    F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))
)

# Adding date_id (integer key for joining fact table)
df_silver = df_silver.withColumn("date_id", F.date_format(F.col("date"), "yyyyMMdd").cast(IntegerType()))

df_silver.show(5)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_date")

print("slv_date written:", spark.table(f"{catalog_name}.silver.slv_date").count(), "rows")